# 🛍️ Retail Customer Segmentation & Sales Intelligence
### End-to-End Exploratory Data Analysis, Product Intelligence & Unsupervised Clustering

**Objective:**
Analyze the Kaggle UK Online Retail dataset (~540k transactions), net out order cancellations against matching original sales, identify genuine product sales leaders, extract customer behavioral features (RFM+), evaluate three clustering algorithms (**K-Means**, **DBSCAN**, and **Agglomerative Hierarchical Clustering**), profile customer segments, and generate strategic business recommendations.

*All financial figures are in British Pounds (£).*

---


## 1. Setup, Environment & Imports

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is available
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_FILE, RANDOM_STATE
from src.data_loader import load_raw_data
from src.cleaning import clean_retail_data, get_customer_clustering_data
from src.product_analysis import analyze_products
from src.features import compute_customer_features, prepare_features_for_clustering
from src.clustering import run_kmeans_search, fit_kmeans, run_dbscan_kdistance, fit_dbscan, fit_agglomerative, compute_pca_projection, assign_kmeans_segment_names
from src.evaluation import compute_clustering_metrics, compute_segment_profiles, profile_dbscan_noise, compute_ari_matrix
from src.insights import compute_segment_product_insights, cluster_products, run_market_basket_analysis

print("Environment configured successfully. Random seed:", RANDOM_STATE)


Environment configured successfully. Random seed: 42


## 2. Data Ingestion & Cancellation Netting
The raw dataset contains 541,909 records. We enforce rigorous cleaning:
1. Parse datetime & compute initial Revenue = Quantity * UnitPrice
2. Drop exact duplicates
3. Remove non-product StockCodes (POST, D, M, BANK CHARGES, etc.)
4. **Cancellation Netting**: Match each cancellation against earlier original sales with identical CustomerID and StockCode:
   - Full cancellation: original sale is completely removed.
   - Partial cancellation: original sale quantity is reduced.
5. Filter valid positive Quantity and UnitPrice
6. Standardize descriptions to the modal name per StockCode


In [2]:
df_raw = load_raw_data(RAW_DATA_FILE)
df_cleaned, counts_log = clean_retail_data(df_raw, save_processed=False)

print("\nCleaning & Netting Funnel:")
for step, count in counts_log.items():
    if "revenue" in step:
        print(f"  {step}: £{count:,.2f}")
    else:
        print(f"  {step}: {count:,} records")


2026-11-24 14:22:23,567 [INFO] Loading raw data from /Users/dakshjoshi/Desktop/PROJECT-1/data/raw/data.csv...


2026-11-24 14:22:23,639 [INFO] Successfully loaded 541,909 rows and 8 columns.


2026-11-24 14:22:23,639 [INFO] [Step 0] Initial raw records: 541,909


2026-11-24 14:22:23,641 [INFO] [Step 1] Parsing InvoiceDate and computing initial Revenue...


2026-11-24 14:22:23,687 [INFO] [Step 1] Records after datetime parsing: 541,909


2026-11-24 14:22:23,761 [INFO] [Step 2] Dropped 5,268 exact duplicates. Remaining: 536,641


... (4 outputs left out)

2026-11-24 14:22:31,040 [INFO] [Step 5] Valid sales after cancellation netting: 519,879


2026-11-24 14:22:31,040 [INFO] [Step 6] Standardizing StockCode Descriptions...


2026-11-24 14:22:31,132 [INFO] [Step 6] Final cleaned records count: 519,879



Cleaning & Netting Funnel:
  0_initial_rows: 541,909 records
  1_after_date_parse: 541,909 records
  2_after_drop_duplicates: 536,641 records
  3_after_remove_non_products: 533,805 records
  4_netting_original_rows_removed: 2,766 records
  4_netting_original_rows_reduced: 4,826 records
  4_netting_revenue_gbp: £485,586.01
  5_after_netting_and_valid_sales: 519,879 records
  6_final_cleaned_rows: 519,879 records


## 3. Product Sales & Performance Analysis
After netting out cancellations, we determine the **genuine Most Sold Product** (by units) and contrast it against the revenue and cart order leaders.


In [3]:
product_results = analyze_products(df_cleaned, save_csv=False)

most_sold_units = product_results["most_sold_by_units"]
top_rev = product_results["top_by_revenue"]
top_ord = product_results["top_by_orders"]

print(f"🏆 Most Sold Product (Units): '{most_sold_units['Description']}' ({most_sold_units['TotalUnits']:,} units across {most_sold_units['DistinctOrders']:,} orders and {most_sold_units['DistinctCustomers']:,} customers)")
print(f"💰 Top Product by Revenue: '{top_rev['Description']}' (£{top_rev['TotalRevenue']:,.2f} across {top_rev['DistinctOrders']:,} orders)")
print(f"🛒 Top Product by Orders: '{top_ord['Description']}' ({top_ord['DistinctOrders']:,} distinct orders)")

# Display top 5 by units
product_results["top10_units"][["StockCode", "Description", "TotalUnits", "TotalRevenue", "DistinctOrders", "DistinctCustomers"]].head(5)


2026-11-24 14:22:31,136 [INFO] Computing product-level statistics...


2026-11-24 14:22:31,268 [INFO] Most Sold Product (by Units): 'POPCORN HOLDER' (StockCode: 22197) with 56,432 units sold.


2026-11-24 14:22:31,268 [INFO] Top Product by Revenue: 'REGENCY CAKESTAND 3 TIER' (StockCode: 22423) with £165,378.99 revenue.


2026-11-24 14:22:31,268 [INFO] Top Product by Orders: 'WHITE HANGING HEART T-LIGHT HOLDER' (StockCode: 85123A) with 2,188 orders.


🏆 Most Sold Product (Units): 'POPCORN HOLDER' (56,432 units across 1,389 orders and 406 customers)
💰 Top Product by Revenue: 'REGENCY CAKESTAND 3 TIER' (£165,378.99 across 1,949 orders)
🛒 Top Product by Orders: 'WHITE HANGING HEART T-LIGHT HOLDER' (2,188 distinct orders)


,StockCode,Description,TotalUnits,TotalRevenue,DistinctOrders,DistinctCustomers
1107,22197,POPCORN HOLDER,56432,50987.12,1389,406
2905,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,53751,13542.81,531,305
3382,85099B,JUMBO BAG RED RETROSPOT,47350,92338.17,2072,633
3189,84879,ASSORTED COLOUR BIRD ORNAMENT,36316,58851.56,1452,677
438,21212,PACK OF 72 RETROSPOT CAKE CASES,36157,21115.26,1314,630


## 4. Customer Behavioral Feature Engineering (RFM + Extras)
For customer clustering, records without `CustomerID` are dropped. We calculate:
- **Recency**: Days since last transaction (Snapshot date = max date + 1)
- **Frequency**: Distinct invoice count
- **Monetary**: Total spend in GBP (£)
- **AvgOrderValue**: Monetary / Frequency
- **UniqueProducts**: Distinct StockCodes purchased
- **AvgBasketSize**: Total units / Frequency

Features are clipped at the 99.5th percentile to limit extreme leverage, log1p-transformed to address right skew, and standardized via `StandardScaler`.


In [4]:
df_cust_raw = get_customer_clustering_data(df_cleaned)
features_df = compute_customer_features(df_cust_raw)
features_df, df_log, X_scaled, scaler = prepare_features_for_clustering(features_df, save_parquet=False)

print(f"Total Unique Active Customers: {len(features_df):,}")
print("Extracted Features Summary:")
features_df[["Recency", "Frequency", "Monetary", "AvgOrderValue", "UniqueProducts", "AvgBasketSize"]].describe().round(2)


2026-11-24 14:22:31,637 [INFO] Filtered for customer clustering: 519,879 -> 388,517 records (CustomerID present).


2026-11-24 14:22:31,659 [INFO] Computing customer features with snapshot date: 2011-12-10 12:50:00


2026-11-24 14:22:31,844 [INFO] Generated features for 4,324 unique active customers.


2026-11-24 14:22:31,844 [INFO] Preparing features for clustering across columns: ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts', 'AvgBasketSize']


2026-11-24 14:22:31,856 [INFO] Scaled feature matrix shape: (4324, 6)


Total Unique Active Customers: 4,324
Extracted Features Summary:


,Recency,Frequency,Monetary,AvgOrderValue,UniqueProducts,AvgBasketSize
count,4324.00,4324.00,4324.00,4324.00,4324.00,4324.00
mean,92.83,4.23,1909.88,370.32,61.23,226.73
std,100.26,7.57,8303.32,454.40,84.95,306.89
min,1.00,1.00,2.90,2.90,1.00,1.00
25%,18.00,1.00,299.98,173.85,16.00,92.00
50%,51.00,2.00,652.77,284.71,35.00,159.65
75%,143.00,5.00,1610.46,422.46,77.00,269.00
max,374.00,202.00,278742.02,12425.45,1756.00,7824.00


## 5. Unsupervised Clustering Algorithms
We compare three distinct clustering families:
1. **K-Means (Partition-based)**: Evaluated across k=2..10 using Elbow Inertia, Silhouette, and Calinski-Harabasz.
2. **DBSCAN (Density-based)**: k-distance plot knee detection with hyperparameter grid search.
3. **Agglomerative Hierarchical**: Comparison of Ward, Complete, and Average linkages.


In [5]:
# 5.1 K-Means Search
km_search = run_kmeans_search(X_scaled)
print(f"Optimal Elbow k detected: {km_search['elbow_k']}")

# Fit K-Means
selected_k = 5
km_labels, km_model, km_runtime = fit_kmeans(X_scaled, n_clusters=selected_k)
km_metrics = compute_clustering_metrics(X_scaled, km_labels, runtime_sec=km_runtime, model_name=f"K-Means (k={selected_k})")

# 5.2 DBSCAN
k_distances, knee_eps = run_dbscan_kdistance(X_scaled, k=10)
db_labels, db_model, db_runtime = fit_dbscan(X_scaled, eps=1.0, min_samples=5)
db_metrics = compute_clustering_metrics(X_scaled, db_labels, runtime_sec=db_runtime, model_name="DBSCAN (eps=1.0, ms=5)")

# 5.3 Agglomerative
agg_labels, agg_model, agg_runtime = fit_agglomerative(X_scaled, n_clusters=selected_k, linkage="ward")
agg_metrics = compute_clustering_metrics(X_scaled, agg_labels, runtime_sec=agg_runtime, model_name=f"Agglomerative (Ward, k={selected_k})")

# Comparison Table
model_comp_df = pd.DataFrame([km_metrics, db_metrics, agg_metrics])
model_comp_df


Optimal Elbow k detected: 5


,Model,Clusters,Noise %,Silhouette,Davies-Bouldin,Calinski-Harabasz,Runtime (s)
0,K-Means (k=5),5,0.00,0.2252,1.2333,1752.80,0.107
1,"DBSCAN (eps=1.0, ms=5)",2,3.03,0.3799,0.5782,24.62,0.055
2,"Agglomerative (Ward, k=5)",5,0.00,0.1380,1.6408,1318.02,0.121


## 6. Segment Profiling & Pareto Analysis
We name segments empirically from centroid rankings:
- **Champions**: Highest spend, highest frequency, lowest recency
- **Potential Loyalists**: Recent, high-volume transactions
- **Loyal**: Consistent repeat buyers
- **Lost / Hibernating**: Lowest spend, longest elapsed recency
- **At-Risk**: High past spend, but inactive recently


In [6]:
df_segmented = assign_kmeans_segment_names(features_df, km_labels)
profiles = compute_segment_profiles(df_segmented, cluster_col="Segment")
profiles[["Segment", "CustomerCount", "PctCustomers", "TotalRevenue", "PctRevenue", "MeanRecency", "MeanFrequency", "MeanMonetary"]].round(2)


,Segment,CustomerCount,PctCustomers,TotalRevenue,PctRevenue,MeanRecency,MeanFrequency,MeanMonetary
0,At-Risk,1162,26.87,447197.20,5.42,157.58,1.39,384.85
1,Champions,689,15.93,5332177.25,64.57,15.42,13.61,7739.01
2,Lost / Hibernating,663,15.33,111620.95,1.35,158.21,1.56,168.36
3,Loyal,1016,23.50,920256.30,11.14,32.65,4.10,905.76
4,Potential Loyalists,794,18.36,1447075.93,17.52,87.65,2.61,1822.51


## 7. Business Insights & Association Rules
Segment-specific product affinities, product catalog clustering, and Market Basket Analysis.


In [7]:
seg_insights = compute_segment_product_insights(df_cleaned, df_segmented)
print("Top Product per Customer Segment:")
display(seg_insights["top_product_per_segment"])

print("\nPareto Distribution Summary:")
display(seg_insights["pareto_summary"].round(2))


2026-11-24 14:22:34,424 [INFO] Computing segment-specific product affinities and revenue shares...


Top Product per Customer Segment:


,Segment,StockCode,Description,UnitsPurchased,SegmentRevenue
0,At-Risk,18007,ESSENTIAL BALM 3.5G TIN IN ENVELOPE,5586,335.16
1,Champions,22197,POPCORN HOLDER,37799,28467.57
2,Lost / Hibernating,84946,ANTIQUE SILVER T-LIGHT GLASS,1052,1232.92
3,Loyal,84879,ASSORTED COLOUR BIRD ORNAMENT,4276,7149.64
4,Potential Loyalists,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,13777,3379.01



Pareto Distribution Summary:


,Segment,CustomerCount,TotalRevenue,CustomerSharePct,RevenueSharePct
1,Champions,689,5332177.25,15.93,64.57
4,Potential Loyalists,794,1447075.93,18.36,17.52
3,Loyal,1016,920256.30,23.50,11.14
0,At-Risk,1162,447197.20,26.87,5.42
2,Lost / Hibernating,663,111620.95,15.33,1.35


## 8. 2D PCA Customer Projections
Visualizing the customer feature space using Principal Component Analysis.


In [8]:
X_pca = compute_pca_projection(X_scaled, n_components=2)
df_segmented["PCA1"] = X_pca[:, 0]
df_segmented["PCA2"] = X_pca[:, 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_segmented,
    x="PCA1",
    y="PCA2",
    hue="Segment",
    palette="Set2",
    alpha=0.8
)
plt.title("2D PCA Customer Segmentation (K-Means)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


2026-11-24 14:22:34,493 [INFO] PCA explained variance ratio: [0.58249443 0.23468413] (Total: 81.72%)


/var/folders/p6/xr1vzx4152v1gmj98mvhdj9m0000gn/T/ipykernel_5604/3586872876.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Conclusion
- **Cancellation Netting:** Successfully removed 2,766 fully cancelled orders and reduced 4,826 partially cancelled sales, netting out £485,586.01 in artificial revenue.
- **Genuine Most Sold Product:** `POPCORN HOLDER` leads in units sold (56,432 units across 1,389 orders and 406 customers). `REGENCY CAKESTAND 3 TIER` generates the highest revenue (£165,378.99 across 1,949 orders), and `WHITE HANGING HEART T-LIGHT HOLDER` is the most frequently ordered across carts (2,188 orders).
- **Best Clustering Model:** **K-Means (k=5)** achieves the highest Calinski-Harabasz score (1717.3) with distinct operational cohorts.
- **Pareto Principle:** Champions (15.9% of customers) generate **64.6%** of all revenue, underscoring the critical need for dedicated VIP retention.
